# Profile-RAG — Half 2: Predict (Llama 3 8B · Kaggle)

Kaggle version of `notebook/gpt/rag_profile_half2_predict.ipynb`.
Uses `meta-llama/Meta-Llama-3-8B-Instruct` loaded once in 4-bit via bitsandbytes.

**Requires:** `data/vector_db/myp_profile/` from Half 1 (already committed to the repo).

Pipeline:
1. Clone repo & install deps
2. Login to HuggingFace
3. Load Llama 3 8B once — patch `hf_call` and profiler's `gpt_call` to use the resident model
4. Profile the test set (label-blind) → `data/profile_db/myp_test/`
5. Install `ProfileRAGRetriever`
6. Sanity check
7. `predict`
8. `evaluate`
9. Cleanup

## Setup — clone repo & install dependencies

In [ ]:
!git clone https://github.com/mtrung12/model_x_ocean.git
%cd /kaggle/working/model_x_ocean/

In [ ]:
!pip install -U transformers accelerate sentencepiece safetensors bitsandbytes faiss-cpu
!pip install -r requirements.txt

## HuggingFace login (Kaggle secret: `llama-3-8B`)

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
token = user_secrets.get_secret("llama-3-8B")
login(token=token)

## Imports

In [ ]:
from pathlib import Path
import sys, os, json
from typing import Dict

import numpy as np
import pandas as pd

project_root = Path("/kaggle/working/model_x_ocean")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from rag.profiler.store import ProfileStore
from rag.profiler.runner import build_profiles
from rag.profiler.prompts import (
    FACETS,
    slice_profile_for_trait,
)
from rag.embedder import get_embedding
import rag.retriever as _retriever_mod
from rag.retriever import FeatureRAGRetriever

from ptd_model.predict import predict
from ptd_model.evaluate import evaluate

print("Project root:", project_root)

## Configuration

In [ ]:
# --- Paths ----------------------------------------------------------------
test_csv        = str(project_root / "data/split/myp/test.csv")

test_profile_db = str(project_root / "data/profile_db/myp_test")     # generated below if missing
vector_db_dir   = str(project_root / "data/vector_db/myp_profile")   # built by Half 1

res_dir         = str(project_root / "result")
log_dir         = str(project_root / "log")

# --- Model ----------------------------------------------------------------
HF_MODEL_ID     = "meta-llama/Meta-Llama-3-8B-Instruct"
model_name      = "meta-llama/Meta-Llama-3-8B-Instruct"
max_new_tokens  = 384   # reduced from 512 to save KV-cache memory
temperature     = 0.0

# Max total sequence length (input + output). Truncates long prompts to fit.
MAX_SEQ_LEN     = 2048

# --- Prompt mode ----------------------------------------------------------
prompt_mode     = "reasoned_rag_def_oneshot"
top_k           = 3    # reduced from 5; each RAG slice is ~200 tokens

TRAIT_NAMES = ("Openness to Experience", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism")
TRAIT_CODES = {
    "Openness to Experience": "cOPN",
    "Conscientiousness":      "cCON",
    "Extraversion":           "cEXT",
    "Agreeableness":          "cAGR",
    "Neuroticism":            "cNEU",
}

test_df = pd.read_csv(test_csv)
print(f"Test  : {len(test_df):>4} rows  | mode={prompt_mode}  | top_k={top_k}")

# Confirm vector DB from Half 1 is present
for fname in ("vectors.faiss", "vectors_meta.jsonl"):
    p = Path(vector_db_dir) / fname
    if not p.exists():
        raise RuntimeError(f"Missing {p}. Run rag_profile_half1_embed.ipynb first.")
print(f"Vector DB found at: {vector_db_dir}")

## Load Llama 3 8B once — patch all LLM call sites

The default `hf_call` reloads the model on every single call, which would take hours.
Here we load the model once and monkey-patch both:
- `utils.hf_client.hf_call` — used by `ptd_model.predict` for classification
- `utils.gpt_client.gpt_call` — used by `rag.profiler.runner` for test-set profiling

Both are routed through the same resident pipeline.

In [ ]:
import os
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Reduces fragmentation — recommended by PyTorch for long-running inference
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

print(f"Loading {HF_MODEL_ID} in 4-bit ...")

_bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
_tokenizer.pad_token = _tokenizer.eos_token
_model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_ID,
    quantization_config=_bnb_cfg,
    device_map="auto",
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
_model.eval()
print("Model loaded.")


def _resident_llama_call(
    user_prompt: str,
    system_prompt: str,
    model: str,          # ignored — always uses the resident model
    max_new_tokens: int,
    temperature: float,
) -> str:
    """Call the resident Llama model without reloading weights.

    Truncates the input to MAX_SEQ_LEN - max_new_tokens tokens so that
    input + output never exceeds MAX_SEQ_LEN, preventing KV-cache OOM.
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    prompt = _tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    max_input_len = MAX_SEQ_LEN - max_new_tokens
    inputs = _tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_len,
    ).to(_model.device)
    input_length = inputs["input_ids"].shape[-1]

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        pad_token_id=_tokenizer.pad_token_id,
    )
    if temperature > 0:
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = temperature
    else:
        gen_kwargs["do_sample"] = False

    with torch.no_grad():
        outputs = _model.generate(**inputs, **gen_kwargs)

    generated_tokens = outputs[0][input_length:]
    del inputs, outputs
    torch.cuda.empty_cache()
    gc.collect()
    return _tokenizer.decode(generated_tokens, skip_special_tokens=True)


# Patch hf_client so predict() never reloads the model
import utils.hf_client as _hf_mod
_hf_mod.hf_call = _resident_llama_call

# Patch gpt_call so the profiler runner (which hardcodes gpt_call) also uses Llama
import utils.gpt_client as _gpt_mod
_orig_gpt_call = _gpt_mod.gpt_call
_gpt_mod.gpt_call = _resident_llama_call

# Re-import runner so it picks up the patched gpt_call
import importlib
import rag.profiler.runner as _runner_mod
importlib.reload(_runner_mod)
from rag.profiler.runner import build_profiles

print("[patch] hf_call and gpt_call -> resident Llama 3 8B")

## Step 1 — Profile the test set (label-blind)

Test profiles MUST be generated **without exposing labels** (`use_labels=False`)
to avoid leakage: at inference time the model would not know the ground truth.

In [ ]:
test_store_path = Path(test_profile_db) / "profile_store.jsonl"
test_store = ProfileStore(str(test_store_path))
test_store.load()
needed = len(test_df) - sum(
    1 for i in range(len(test_df)) if test_store.has(f"user_{i}") and test_store.get(f"user_{i}").get("valid")
)
print(f"Test profiles already in store: {len(test_store)}; missing: {needed}")

if needed > 0:
    test_store = _runner_mod.build_profiles(
        data       = test_df,
        output_dir = test_profile_db,
        model_name = model_name,   # ignored by patched gpt_call — uses resident Llama
        log_dir    = str(Path(log_dir) / "profiler_test"),
        use_labels = False,        # IMPORTANT: label-blind for test
    )
test_entries_by_idx = {
    int(e["user_id"].split("_")[1]): e for e in test_store.get_all() if e.get("valid")
}
print(f"Test profiles ready: {len(test_entries_by_idx)}")

In [ ]:
# Flush GPU cache before prediction — profiling allocates temporary tensors
# that may not be released until an explicit empty_cache() call.
import torch, gc
torch.cuda.empty_cache()
gc.collect()
print(f"GPU free after profiling: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GiB")

## Step 2 — Profile-aware retriever adapter

Subclasses `FeatureRAGRetriever` to:
- embed the **test essay's profile** (not the raw essay) as the query,
- render retrieved exemplars as **trait-sliced profiles + label**.

Monkey-patched onto `rag.retriever.FeatureRAGRetriever` so that
`ptd_model.predict` picks it up without any changes to predictor code.

In [ ]:
def render_full_profile_text(entry: Dict) -> str:
    """Deterministic rendering of a profile for embedding."""
    raw = entry.get("raw") or ""
    if raw.strip():
        return raw
    facets = entry.get("facets", {})
    ling   = entry.get("linguistic", {})
    lines = ["[FACETS]"]
    for code, name, *_ in FACETS:
        f = facets.get(code, {})
        lines.append(f"{code} {name:<18}| {f.get('signal','')} | {f.get('evidence','')}")
    lines.append("\n[LINGUISTIC]")
    for k, v in ling.items():
        lines.append(f"{k}: {v}")
    return "\n".join(lines)


class ProfileRAGRetriever(FeatureRAGRetriever):
    """Retriever that embeds the query essay's parsed profile (not raw text)
    and returns trait-sliced profile excerpts as few-shot exemplars.
    """

    def __init__(self, db_dir: str, test_profiles_by_idx: Dict[int, Dict], test_df: pd.DataFrame):
        super().__init__(db_dir=db_dir)
        self._use_finetuned = False
        self._test_profiles_by_idx = test_profiles_by_idx
        self._text_to_idx = {str(t): i for i, t in enumerate(test_df["text"].tolist())}

    def _embed_query_profile(self, query_text: str):
        idx = self._text_to_idx.get(str(query_text))
        if idx is None or idx not in self._test_profiles_by_idx:
            print(f"  [retriever] WARN: no test profile found for query (idx={idx}); falling back to raw text.")
            return self._embed_query(query_text)
        profile_text = render_full_profile_text(self._test_profiles_by_idx[idx])
        return np.array(self._embed_query(profile_text), dtype="float32")

    def build_similar_context(self, posts: str, trait: str, top_k: int = 3) -> str:
        trait_code = TRAIT_CODES.get(trait)
        if trait_code is None:
            return super().build_similar_context(posts=posts, trait=trait, top_k=top_k)

        query_emb = self._embed_query_profile(posts)
        all_results = self._search(query_emb, top_k * 4)

        blocks, seen = [], 0
        for r in all_results:
            if trait not in r.get("trait_labels", {}):
                continue
            label = r["trait_labels"][trait]
            features = r.get("features", {}) or {}
            profile = features.get("profile") or {}
            slice_text = slice_profile_for_trait(profile, trait_code) if profile else ""
            if not slice_text.strip():
                slice_text = "  (no profile slice available)"
            blocks.append(
                f"[Similar Profile {seen+1}] (label: {label})\n{slice_text}"
            )
            seen += 1
            if seen >= top_k:
                break
        return "\n\n".join(blocks)

In [ ]:
# Monkey-patch so ptd_model.predict instantiates the profile-aware retriever
_OriginalRetriever = _retriever_mod.FeatureRAGRetriever

def _RetrieverFactory(db_dir=None):
    return ProfileRAGRetriever(
        db_dir=db_dir or vector_db_dir,
        test_profiles_by_idx=test_entries_by_idx,
        test_df=test_df,
    )

_retriever_mod.FeatureRAGRetriever = _RetrieverFactory
print("[adapter] ProfileRAGRetriever installed.")

## Step 3 — Sanity check: render one few-shot context

Confirms the retriever round-trips correctly for the first test essay.

In [ ]:
_smoke = _RetrieverFactory(db_dir=vector_db_dir)
_query_text = test_df.iloc[0]["text"]
for trait_full in TRAIT_NAMES:
    print(f"\n=== {trait_full} ===")
    print(_smoke.build_similar_context(posts=_query_text, trait=trait_full, top_k=top_k))

## Step 4 — Run prediction

In [ ]:
run_id, run_time, prediction_csv = predict(
    text_df        = test_df,
    model_name     = model_name,
    log_dir        = log_dir,
    prompt_mode    = prompt_mode,
    max_new_tokens = max_new_tokens,
    res_dir        = res_dir,
    temperature    = temperature,
    top_k          = top_k,
    vector_db_dir  = vector_db_dir,
)

print(f"\nDone in {run_time:.1f}s")
print(f"Predictions saved to: {prediction_csv}")

## Step 5 — Evaluate

In [ ]:
evaluation = evaluate(
    prediction_csv = prediction_csv,
    model_name     = model_name,
    res_dir        = res_dir,
    run_time       = run_time,
    prompt_mode    = prompt_mode,
    run_id         = run_id,
)

print("Summary CSV:", evaluation["summary_csv"])
print(f"Failed predictions: {evaluation['fail_count']} / {evaluation['n_records']}")
summary_df = pd.read_csv(evaluation["summary_csv"])
display(summary_df[["trait", "n_samples", "accuracy", "macro_f1", "weighted_f1"]]
        .sort_values("accuracy", ascending=False)
        .reset_index(drop=True))

## Step 6 — Restore original retriever (cleanup)

In [ ]:
_retriever_mod.FeatureRAGRetriever = _OriginalRetriever
_gpt_mod.gpt_call = _orig_gpt_call
print("[adapter] Original FeatureRAGRetriever and gpt_call restored.")